<div style="background-color:#EAEAEA; padding:20px; border-left:5px solid #6C757D; border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;"><td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1><hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de València</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px; font-weight:700; line-height:1.3; margin-top:14px; margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan — <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente — <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none; width:230px; padding-left:25px; text-align:right; vertical-align:top;"><img src="../../../../logo_uv.png" alt="Universitat de València" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Borexino 2018 Open Data — Online Data EDA

This notebook reads the provider's public online resources directly, without using the converted TPeanuts tables. It inventories the remote content, checks formats and units, and determines which canonical products can be generated.

Official source: [https://borex.lngs.infn.it/papers/articles/solar_nu/comprehensive-measurement-of-pp-chain-solar-neutrinos/](https://borex.lngs.infn.it/papers/articles/solar_nu/comprehensive-measurement-of-pp-chain-solar-neutrinos/)

## Available products

- Detector observations: low-energy reconstructed spectrum and high-energy radial distribution.
- Probability: experimental Pee points for pp, 7Be, pep and 8B.
- Fluxes/rates: publication fit results; not solar-model predictions.
- Density, radial production and production spectra: unavailable.

## 1. Imports and remote access

In [1]:
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import json
import re
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TPeanuts project root")


def fetch(url: str, *, timeout: int = 60) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "TPeanuts/solar-data"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download(url: str, path: Path, *, overwrite: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or not path.exists():
        path.write_bytes(fetch(url))
    return path


PROJECT_ROOT = find_project_root(Path.cwd())
SOLAR_DATA = PROJECT_ROOT / "data" / "solar"

## 2. Online inventory and format inspection

In [2]:
URLS = {"low_energy_spectrum": "https://borex.lngs.infn.it/wp-content/uploads/OpenData/Nature2018_Fig2a_DATA.txt", "high_energy_radial": "https://borex.lngs.infn.it/wp-content/uploads/OpenData/Nature2018_Fig2b_DATA.txt", "survival_probability": "https://borex.lngs.infn.it/wp-content/uploads/OpenData/Nature2018_Fig3_DATApoints.txt"}
rows, remote = [], {}
for name, url in URLS.items():
    payload = fetch(url)
    remote[name] = payload.decode("utf-8", errors="replace")
    rows.append({"resource": name, "bytes": len(payload), "sha256": hashlib.sha256(payload.encode() if isinstance(payload, str) else payload).hexdigest(), "url": url})
display(pd.DataFrame(rows))
print(remote["survival_probability"])

,resource,bytes,sha256,url
0,low_energy_spectrum,45720,82be5552fc141c8b0f9469bde199b6b8754433c4b24ae8...,https://borex.lngs.infn.it/wp-content/uploads/...
1,high_energy_radial,1497,e1717161fe5590fe581f1fb26f3feb4c2eb44865299953...,https://borex.lngs.infn.it/wp-content/uploads/...
2,survival_probability,305,391b3438c75fdab0cb933e84f0f56156bb2230bd9059dd...,https://borex.lngs.infn.it/wp-content/uploads/...


pp 			
Energy [MeV]	Pee	Err Up	Err Down
0.267		0.566	0.092	0.092
			
Be7			
Energy [MeV]	Pee	Err Up	Err Down
0.862		0.532	0.054	0.054
			
pep			
Energy [MeV]	Pee	Err Up	Err Down
1.44		0.428	0.114	0.114
			
B8			
Energy [MeV]	Pee	Err Up	Err Down
7.4		0.39	0.09	0.09
8.1		0.37	0.08	0.08
9.7		0.35	0.09	0.09



## 3. Conclusions

Only source-published observables are attributed to this provider. Derived TPeanuts quantities and detector-level spectra are labelled separately by the generator notebook.